SCENARIO 1

PROMPT

You are a financial data analyst responsible for analyzing inventory turnover and stock movement patterns. I have a dataset with the following fields: Product_Id, Category, Stock_level, last_Restock_date, Sales_last_30_days, supplier_lead_time, and unit_cost. Generate a complete Python script that loads and cleans the data, validates that all required columns exist, fills missing numeric values using medians, calculates inventory turnover rates, identifies slow moving products, predicts potential stockouts, ensures all values are within expected ranges, and creates data visualizations. Use the Python libraries Pandas, Numpy and matplotlib.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import logging

# =====================================================
# CONFIGURE LOGGING
# =====================================================

logging.basicConfig(
    filename='inventory_analysis.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# =====================================================
# LOAD DATA
# =====================================================

try:
    df = pd.read_csv("inventory.csv")
    logging.info("Inventory file loaded successfully.")

except FileNotFoundError:
    raise FileNotFoundError(
        "inventory.csv was not found."
    )

except pd.errors.EmptyDataError:
    raise ValueError(
        "inventory.csv is empty."
    )

except Exception as e:
    raise RuntimeError(
        f"Unexpected error while loading file: {e}"
    )

# =====================================================
# VALIDATE REQUIRED COLUMNS
# =====================================================

required_columns = [
    "Product_Id",
    "Category",
    "Stock_level",
    "last_Restock_date",
    "Sales_last_30_days",
    "supplier_lead_time",
    "unit_cost"
]

missing_columns = (
    set(required_columns)
    - set(df.columns)
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# =====================================================
# CLEAN NUMERIC FIELDS
# =====================================================

numeric_columns = [
    "Stock_level",
    "Sales_last_30_days",
    "supplier_lead_time",
    "unit_cost"
]

for column in numeric_columns:

    original_missing = df[column].isna().sum()

    # Convert strings such as "abc" to NaN
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    new_missing = df[column].isna().sum()

    if new_missing > original_missing:
        logging.warning(
            f"{column}: Invalid values converted to NaN."
        )

    median_value = df[column].median()

    df[column] = df[column].fillna(
        median_value
    )

# =====================================================
# CLEAN CATEGORICAL FIELDS
# =====================================================

df["Category"] = (
    df["Category"]
    .fillna("Unknown")
    .astype(str)
)

df["Product_Id"] = (
    df["Product_Id"]
    .fillna("Missing_ID")
    .astype(str)
)

# =====================================================
# CLEAN DATE FIELD
# =====================================================

df["last_Restock_date"] = pd.to_datetime(
    df["last_Restock_date"],
    errors="coerce"
)

invalid_dates = (
    df["last_Restock_date"]
    .isna()
    .sum()
)

if invalid_dates > 0:
    logging.warning(
        f"{invalid_dates} invalid date values found."
    )

# Replace invalid dates with median date

median_date = (
    df["last_Restock_date"]
    .dropna()
    .median()
)

df["last_Restock_date"] = (
    df["last_Restock_date"]
    .fillna(median_date)
)

# =====================================================
# RANGE VALIDATION
# =====================================================

df["Stock_level"] = (
    df["Stock_level"]
    .clip(lower=0)
)

df["Sales_last_30_days"] = (
    df["Sales_last_30_days"]
    .clip(lower=0)
)

df["supplier_lead_time"] = (
    df["supplier_lead_time"]
    .clip(lower=0)
)

df["unit_cost"] = (
    df["unit_cost"]
    .clip(lower=0.01)
)

# =====================================================
# DATE CALCULATIONS
# =====================================================

today = pd.Timestamp.today()

df["days_since_restock"] = (
    today -
    df["last_Restock_date"]
).dt.days

# =====================================================
# INVENTORY TURNOVER
# =====================================================

df["Inventory_Turnover"] = np.where(
    df["Stock_level"] > 0,
    df["Sales_last_30_days"] /
    df["Stock_level"],
    0
)

# =====================================================
# DAILY DEMAND
# =====================================================

df["Daily_Sales"] = (
    df["Sales_last_30_days"] / 30
)

# =====================================================
# DAYS OF INVENTORY REMAINING
# =====================================================

df["Days_Remaining"] = np.where(
    df["Daily_Sales"] > 0,
    df["Stock_level"] /
    df["Daily_Sales"],
    np.inf
)

# =====================================================
# IDENTIFY SLOW-MOVING PRODUCTS
# =====================================================

df["Slow_Moving"] = (
    (df["Inventory_Turnover"] < 0.20)
    |
    (df["Sales_last_30_days"] == 0)
)

# =====================================================
# PREDICT POTENTIAL STOCKOUTS
# =====================================================

df["Potential_Stockout"] = (
    df["Days_Remaining"]
    <
    df["supplier_lead_time"]
)

# =====================================================
# SUMMARY STATISTICS
# =====================================================

total_products = len(df)

slow_moving_count = (
    df["Slow_Moving"]
    .sum()
)

stockout_count = (
    df["Potential_Stockout"]
    .sum()
)

print("\n===== INVENTORY ANALYSIS SUMMARY =====")

print(f"Total Products: {total_products}")

print(
    f"Slow Moving Products: "
    f"{slow_moving_count}"
)

print(
    f"Potential Stockouts: "
    f"{stockout_count}"
)

# =====================================================
# SAVE RESULTS
# =====================================================

df.to_csv(
    "inventory_analysis_results.csv",
    index=False
)

logging.info(
    "Analysis results exported successfully."
)

# =====================================================
# VISUALIZATION
# INVENTORY TURNOVER DISTRIBUTION
# =====================================================

plt.figure(figsize=(10, 6))

plt.hist(
    df["Inventory_Turnover"],
    bins=20
)

plt.title(
    "Inventory Turnover Distribution"
)

plt.xlabel(
    "Inventory Turnover Rate"
)

plt.ylabel(
    "Number of Products"
)

plt.grid(True)

plt.tight_layout()

plt.show()

# =====================================================
# DISPLAY HIGH-RISK PRODUCTS
# =====================================================

print(
    "\n===== POTENTIAL STOCKOUT ITEMS ====="
)

print(
    df.loc[
        df["Potential_Stockout"],
        [
            "Product_Id",
            "Stock_level",
            "Sales_last_30_days",
            "supplier_lead_time",
            "Days_Remaining"
        ]
    ]
)

print(
    "\n===== SLOW MOVING ITEMS ====="
)

print(
    df.loc[
        df["Slow_Moving"],
        [
            "Product_Id",
            "Inventory_Turnover",
            "Stock_level",
            "Sales_last_30_days"
        ]
    ]
)

FOLLOW UP PROMPTS

 I am filling null values in the last_Restock_date column using .median() directly on a datetime Series. Replace this with a correct approach that first drops null values, then sorts the remaining dates chronologically, then picks the middle entry. Do not change any other part of the date cleaning block.

ANOTHER FOLLOW UP PROMPT

Optimise the code by doing the following:
 -wrap the clip operations into a single dictionary loop instead of four separate statements
 -add a duplicate Product_Id check that logs and drops duplicates before analysis runs
 -use unit_cost to calculate Inventory_Value, Revenue_last_30_days and Stockout_Revenue_Risk
 -use days_since_restock to create a Stale_Inventory flag for products unrestocked for over 90 days
 -add boundary validation on dates that rejects future dates and dates before the year 2000 before the median fill runs
 -write all summary statistics to the log file in addition to printing them to console, and replace plt.show() with plt.savefig()
 - add four additional charts covering slow moving products by category, stock level vs days remaining, top ten products by inventory value, and a stockout risk pie chart."

FINAL CODE AFTER FOLLOW UP PROMPTS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import logging

# =====================================================
# CONFIGURE LOGGING
# =====================================================

logging.basicConfig(
    filename='inventory_analysis.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# =====================================================
# LOAD DATA
# =====================================================

try:
    df = pd.read_csv("inventory.csv")
    logging.info("Inventory file loaded successfully.")

except FileNotFoundError:
    raise FileNotFoundError(
        "inventory.csv was not found."
    )

except pd.errors.EmptyDataError:
    raise ValueError(
        "inventory.csv is empty."
    )

except Exception as e:
    raise RuntimeError(
        f"Unexpected error while loading file: {e}"
    )

# =====================================================
# VALIDATE REQUIRED COLUMNS
# =====================================================

required_columns = [
    "Product_Id",
    "Category",
    "Stock_level",
    "last_Restock_date",
    "Sales_last_30_days",
    "supplier_lead_time",
    "unit_cost"
]

missing_columns = (
    set(required_columns)
    - set(df.columns)
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# =====================================================
# DUPLICATE ID CHECK
# Prompt 6 — flag and drop duplicate Product_IDs
# before any analysis runs
# =====================================================

duplicates = df["Product_Id"].duplicated().sum()

if duplicates > 0:
    logging.warning(
        f"{duplicates} duplicate Product_IDs found. "
        f"Keeping first occurrence."
    )
    df = df.drop_duplicates(
        subset="Product_Id",
        keep="first"
    )

# =====================================================
# CLEAN NUMERIC FIELDS
# =====================================================

numeric_columns = [
    "Stock_level",
    "Sales_last_30_days",
    "supplier_lead_time",
    "unit_cost"
]

for column in numeric_columns:

    original_missing = df[column].isna().sum()

    # Convert strings such as "abc" to NaN
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    new_missing = df[column].isna().sum()

    if new_missing > original_missing:
        logging.warning(
            f"{column}: Invalid values converted to NaN."
        )

    median_value = df[column].median()

    df[column] = df[column].fillna(
        median_value
    )

# =====================================================
# CLEAN CATEGORICAL FIELDS
# =====================================================

df["Category"] = (
    df["Category"]
    .fillna("Unknown")
    .astype(str)
)

df["Product_Id"] = (
    df["Product_Id"]
    .fillna("Missing_ID")
    .astype(str)
)

# =====================================================
# CLEAN DATE FIELD
# =====================================================

df["last_Restock_date"] = pd.to_datetime(
    df["last_Restock_date"],
    errors="coerce"
)

# =====================================================
# BOUNDARY VALIDATION ON DATES
# Prompt 8 — reject future dates and impossibly
# old dates before median fill runs
# =====================================================

today = pd.Timestamp.today()

df.loc[
    df["last_Restock_date"] > today,
    "last_Restock_date"
] = pd.NaT

df.loc[
    df["last_Restock_date"] < pd.Timestamp("2000-01-01"),
    "last_Restock_date"
] = pd.NaT

invalid_dates = (
    df["last_Restock_date"]
    .isna()
    .sum()
)

if invalid_dates > 0:
    logging.warning(
        f"{invalid_dates} invalid date values found."
    )

# Replace invalid dates with median date
# Prompt 4 — sort valid dates and pick the true
# positional middle value instead of arithmetic median

# Step 1: isolate and sort all valid (non-null) dates
valid_dates = (
    df["last_Restock_date"]
    .dropna()
    .sort_values()
)

# Step 2: pick the true positional middle value
median_date = valid_dates.iloc[len(valid_dates) // 2]

df["last_Restock_date"] = (
    df["last_Restock_date"]
    .fillna(median_date)
)

# =====================================================
# RANGE VALIDATION
# Prompt 6 — batched into a dictionary loop
# instead of four separate clip statements
# =====================================================

clip_rules = {
    "Stock_level": (0, None),
    "Sales_last_30_days": (0, None),
    "supplier_lead_time": (0, None),
    "unit_cost": (0.01, None)
}

for col, (lower, upper) in clip_rules.items():
    df[col] = df[col].clip(lower=lower, upper=upper)

# =====================================================
# DATE CALCULATIONS
# =====================================================

df["days_since_restock"] = (
    today -
    df["last_Restock_date"]
).dt.days

# =====================================================
# INVENTORY TURNOVER
# =====================================================

df["Inventory_Turnover"] = np.where(
    df["Stock_level"] > 0,
    df["Sales_last_30_days"] /
    df["Stock_level"],
    0
)

# =====================================================
# DAILY DEMAND
# =====================================================

df["Daily_Sales"] = (
    df["Sales_last_30_days"] / 30
)

# =====================================================
# DAYS OF INVENTORY REMAINING
# =====================================================

df["Days_Remaining"] = np.where(
    df["Daily_Sales"] > 0,
    df["Stock_level"] /
    df["Daily_Sales"],
    np.inf
)

# =====================================================
# FINANCIAL METRICS
# Prompt 3 & 6 — unit_cost was cleaned but never
# used. Now drives three financial columns
# =====================================================

df["Inventory_Value"] = (
    df["Stock_level"] * df["unit_cost"]
)

df["Revenue_last_30_days"] = (
    df["Sales_last_30_days"] * df["unit_cost"]
)

df["Stockout_Revenue_Risk"] = (
    df["Daily_Sales"]
    * df["supplier_lead_time"]
    * df["unit_cost"]
)

# =====================================================
# IDENTIFY SLOW-MOVING PRODUCTS
# =====================================================

df["Slow_Moving"] = (
    (df["Inventory_Turnover"] < 0.20)
    |
    (df["Sales_last_30_days"] == 0)
)

# =====================================================
# STALE INVENTORY FLAG
# Prompt 6 & 8 — days_since_restock was calculated
# but never used. Now drives a stale inventory flag
# =====================================================

df["Stale_Inventory"] = (
    (df["days_since_restock"] > 90)
    &
    (df["Slow_Moving"] == True)
)

# =====================================================
# PREDICT POTENTIAL STOCKOUTS
# =====================================================

df["Potential_Stockout"] = (
    df["Days_Remaining"]
    <
    df["supplier_lead_time"]
)

# =====================================================
# SUMMARY STATISTICS
# Prompt 6 — summary now logged to file as well
# as printed to console
# =====================================================

total_products = len(df)
slow_moving_count = df["Slow_Moving"].sum()
stockout_count = df["Potential_Stockout"].sum()
stale_count = df["Stale_Inventory"].sum()
total_inventory_value = df["Inventory_Value"].sum()

print("\n===== INVENTORY ANALYSIS SUMMARY =====")
print(f"Total Products:          {total_products}")
print(f"Slow Moving Products:    {slow_moving_count}")
print(f"Potential Stockouts:     {stockout_count}")
print(f"Stale Inventory Items:   {stale_count}")
print(f"Total Inventory Value:   ${total_inventory_value:,.2f}")

logging.info(f"Total Products: {total_products}")
logging.info(f"Slow Moving Products: {slow_moving_count}")
logging.info(f"Potential Stockouts: {stockout_count}")
logging.info(f"Stale Inventory Items: {stale_count}")
logging.info(
    f"Total Inventory Value: ${total_inventory_value:,.2f}"
)

# =====================================================
# SAVE RESULTS
# =====================================================

df.to_csv(
    "inventory_analysis_results.csv",
    index=False
)

logging.info(
    "Analysis results exported successfully."
)

# =====================================================
# VISUALISATIONS
# Prompt 6 — save figures instead of plt.show()
# and add four additional charts
# =====================================================

# --- Chart 1: Inventory Turnover Distribution -------

plt.figure(figsize=(10, 6))
plt.hist(df["Inventory_Turnover"], bins=20, color="steelblue")
plt.title("Inventory Turnover Distribution")
plt.xlabel("Inventory Turnover Rate")
plt.ylabel("Number of Products")
plt.grid(True)
plt.tight_layout()
plt.savefig("chart1_turnover_distribution.png", dpi=150)
plt.close()

# --- Chart 2: Slow Moving Count by Category ---------

plt.figure(figsize=(10, 6))
(
    df.groupby("Category")["Slow_Moving"]
    .sum()
    .sort_values(ascending=False)
    .plot(kind="bar", color="salmon")
)
plt.title("Slow Moving Products by Category")
plt.xlabel("Category")
plt.ylabel("Number of Slow Moving Products")
plt.xticks(rotation=45, ha="right")
plt.grid(True)
plt.tight_layout()
plt.savefig("chart2_slow_moving_by_category.png", dpi=150)
plt.close()

# --- Chart 3: Stock Level vs Days Remaining ---------

plt.figure(figsize=(10, 6))
plt.scatter(
    df["Stock_level"],
    df["Days_Remaining"].replace(np.inf, 999),
    c=df["Potential_Stockout"].map(
        {True: "red", False: "green"}
    ),
    alpha=0.6
)
plt.title("Stock Level vs Days Remaining")
plt.xlabel("Stock Level")
plt.ylabel("Days Remaining (capped at 999)")
plt.grid(True)
plt.tight_layout()
plt.savefig("chart3_stock_vs_days_remaining.png", dpi=150)
plt.close()

# --- Chart 4: Top 10 Products by Inventory Value ----

plt.figure(figsize=(10, 6))
(
    df.nlargest(10, "Inventory_Value")
    .set_index("Product_Id")["Inventory_Value"]
    .sort_values()
    .plot(kind="barh", color="mediumpurple")
)
plt.title("Top 10 Products by Inventory Value")
plt.xlabel("Inventory Value ($)")
plt.ylabel("Product ID")
plt.grid(True)
plt.tight_layout()
plt.savefig("chart4_top_inventory_value.png", dpi=150)
plt.close()

# --- Chart 5: Stockout Risk Pie Chart ---------------

plt.figure(figsize=(8, 8))
plt.pie(
    [stockout_count, total_products - stockout_count],
    labels=["At Stockout Risk", "Healthy Stock"],
    colors=["tomato", "mediumseagreen"],
    autopct="%1.1f%%",
    startangle=140
)
plt.title("Stockout Risk Overview")
plt.tight_layout()
plt.savefig("chart5_stockout_risk_pie.png", dpi=150)
plt.close()

logging.info("All charts saved successfully.")

# =====================================================
# DISPLAY HIGH-RISK PRODUCTS
# =====================================================

print("\n===== POTENTIAL STOCKOUT ITEMS =====")
print(
    df.loc[
        df["Potential_Stockout"],
        [
            "Product_Id",
            "Stock_level",
            "Sales_last_30_days",
            "supplier_lead_time",
            "Days_Remaining",
            "Stockout_Revenue_Risk"
        ]
    ]
)

print("\n===== SLOW MOVING ITEMS =====")
print(
    df.loc[
        df["Slow_Moving"],
        [
            "Product_Id",
            "Inventory_Turnover",
            "Stock_level",
            "Sales_last_30_days",
            "Inventory_Value"
        ]
    ]
)

print("\n===== STALE INVENTORY ITEMS =====")
print(
    df.loc[
        df["Stale_Inventory"],
        [
            "Product_Id",
            "days_since_restock",
            "Stock_level",
            "Inventory_Value"
        ]
    ]
)

WHAT VISUALISATION APPROACHES WERE SUGGESTED?

1.Inventory Turnover Distribution (Histogram)

Shows how frequently products are selling relative to their stock levels across your entire inventory. A healthy distribution sits toward the right. A spike on the left means most products are sitting unsold, which is a problem worth investigating immediately.

2 Slow Moving Products by Category (Bar Chart)

Breaks down which product categories have the most slow moving items. Instead of just knowing how many slow movers exist overall, this tells you where the problem is concentrated


HOW WAS ERROR HANDLING IMPLEMENTED?

-Missing numbers are filled with the column median so no calculation ever runs on an empty value.
-All warnings and errors are written to a log file throughout so you have a full record of every problem the code encountered when it ran.
-Negative values are caught by the clip operations which force all numeric columns to stay within acceptable minimum ranges.
-Zero division in the turnover and days remaining calculations is caught by np.where which checks for zero stock or zero sales before dividing, returning zero or infinity instead of crashing.
-Impossible dates are caught by boundary checks that reject future dates and anything before year 2000, converting them to NaT before the median fill runs.
-Numeric strings like in number columns are caught which silently converts them to NaN instead of crashing.
-All warnings and errors are written to a log file throughout so you have a full record of every problem the code encountered when it ran.


SCENARIO 2

PROMPT 1
Review the code and improve it

In [ ]:
import pandas as pd


def analyze_user_engagement(logs_df):
    """
    Analyze user engagement metrics.

    Returns:
        summary_metrics (dict)
        device_metrics (DataFrame)
    """

    required_columns = {
        'session_id',
        'page_views',
        'duration',
        'device_type'
    }

    missing = required_columns - set(logs_df.columns)

    if missing:
        raise ValueError(
            f"Missing required columns: {', '.join(missing)}"
        )

    if logs_df.empty:
        raise ValueError("Input dataframe is empty")

    # Clean data
    logs_df = logs_df.copy()

    logs_df['page_views'] = logs_df['page_views'].fillna(0)
    logs_df['duration'] = logs_df['duration'].fillna(0)

    # Session-level metrics
    session_metrics = logs_df.groupby('session_id').agg(
        total_page_views=('page_views', 'sum'),
        total_duration=('duration', 'sum')
    )

    # Bounce rate
    bounce_rate = (
        session_metrics['total_page_views']
        .eq(1)
        .mean()
    )

    summary_metrics = {
        'bounce_rate': round(bounce_rate, 4),
        'avg_session_time':
            session_metrics['total_duration'].mean(),
        'avg_pages_per_session':
            session_metrics['total_page_views'].mean(),
        'total_sessions':
            session_metrics.shape[0]
    }

    # Device metrics
    device_metrics = logs_df.groupby('device_type').agg(
        unique_sessions=('session_id', 'nunique'),
        avg_duration=('duration', 'mean'),
        total_page_views=('page_views', 'sum')
    )

    return summary_metrics, device_metrics

PROMPT 2

Act as a senior Python software engineer and data analyst.

Analyze the code for logical, mathematical, and runtime errors.

Requirements:
- Debug all calculation issues.
- Validate all user inputs and dataset values.
- Use datetime and timedelta for all date/time calculations.
- Handle invalid dates gracefully.
- Add robust exception handling.
- Generate a summary dashboard using a single, most suitable visualization.
- Include clear comments explaining the purpose of each section.
- Optimize the code for maintainability and performance.

For each modification:
1. Identify the original issue.
2. Explain the root cause.
3. Show the fix.
4. Explain how the fix improves reliability.

Output the complete corrected code followed by a detailed technical explanation.

In [ ]:
"""
User Engagement Analytics Module
==================================
Analyzes session-level engagement metrics from raw event logs.
Produces a summary metrics dict, per-device breakdown DataFrame,
and a summary dashboard visualization.

Dependencies: pandas, matplotlib, numpy
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from datetime import datetime, timedelta
import warnings

# ---------------------------------------------------------------------------
# CONSTANTS
# ---------------------------------------------------------------------------
REQUIRED_COLUMNS = {"session_id", "page_views", "duration", "device_type"}
OPTIONAL_DATE_COLUMNS = {"start_time", "end_time"}   # used if present
ROUND_DIGITS = 4


# ---------------------------------------------------------------------------
# SECTION 1 – INPUT VALIDATION
# ---------------------------------------------------------------------------
def _validate_inputs(df: pd.DataFrame) -> pd.DataFrame:
    """
    Validate the structure and content of the incoming DataFrame.

    Checks performed
    ----------------
    1.  Type check – must be a DataFrame.
    2.  Empty check – raises if no rows.
    3.  Required-column check – raises with descriptive list of missing cols.
    4.  Numeric coercion – page_views and duration are cast to numeric;
        non-parseable values become NaN (handled in cleaning step).
    5.  Negative-value guard – page_views / duration must be ≥ 0.
    6.  Optional date parsing – start_time / end_time parsed if present;
        unparseable values are set to NaT rather than crashing.

    Returns a copy of the DataFrame so the original is never mutated.
    """
    # --- 1. Type guard ---
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"Expected a pandas DataFrame, got {type(df).__name__}.")

    # --- 2. Empty guard ---
    if df.empty:
        raise ValueError("Input DataFrame is empty. Cannot compute metrics.")

    # --- 3. Required columns ---
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(
            f"Missing required columns: {', '.join(sorted(missing))}. "
            f"Expected: {sorted(REQUIRED_COLUMNS)}."
        )

    df = df.copy()

    # --- 4. Numeric coercion ---
    # BUG (original): no coercion – string/object columns silently corrupt agg.
    # FIX: errors='coerce' turns non-numeric values into NaN, which are filled
    #      in the cleaning step rather than causing a downstream TypeError.
    for col in ("page_views", "duration"):
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # --- 5. Negative-value guard ---
    # BUG (original): negative durations or page-view counts would skew every
    #      metric (averages, bounce rates) without any warning.
    # FIX: replace negatives with NaN so they are treated as missing data.
    for col in ("page_views", "duration"):
        neg_mask = df[col] < 0
        if neg_mask.any():
            warnings.warn(
                f"Column '{col}' contains {neg_mask.sum()} negative value(s). "
                "These will be treated as missing (NaN).",
                UserWarning,
                stacklevel=3,
            )
            df.loc[neg_mask, col] = np.nan

    # --- 6. Optional datetime parsing ---
    # Use datetime / timedelta for any time-based derivations (requirement).
    for col in OPTIONAL_DATE_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
            n_nat = df[col].isna().sum()
            if n_nat:
                warnings.warn(
                    f"Column '{col}' has {n_nat} unparseable date(s) set to NaT.",
                    UserWarning,
                    stacklevel=3,
                )

    return df


# ---------------------------------------------------------------------------
# SECTION 2 – DATA CLEANING
# ---------------------------------------------------------------------------
def _clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Impute remaining NaN values and derive duration from timestamps when
    the raw duration column is missing or partially null.

    Strategy
    --------
    •  If start_time / end_time exist, compute duration via timedelta where
       the duration column is NaN – this is more reliable than a raw number.
    •  Remaining NaN page_views → 0 (no recorded page view means 0 views).
    •  Remaining NaN duration  → 0 (safest neutral default).
    """
    # Derive duration from timestamps when possible
    # BUG (original): no datetime-based fallback – durations were taken at face
    #      value even when timestamps were available and could be more accurate.
    # FIX: compute timedelta from start_time / end_time and use it to fill gaps.
    if {"start_time", "end_time"}.issubset(df.columns):
        derived = (df["end_time"] - df["start_time"]).dt.total_seconds()
        # Only fill where duration is currently NaN or where derived is valid
        mask = df["duration"].isna() & derived.notna() & (derived >= 0)
        df.loc[mask, "duration"] = derived[mask]

    df["page_views"] = df["page_views"].fillna(0).astype(float)
    df["duration"] = df["duration"].fillna(0).astype(float)

    return df


# ---------------------------------------------------------------------------
# SECTION 3 – METRIC CALCULATIONS
# ---------------------------------------------------------------------------
def _compute_summary_metrics(session_metrics: pd.DataFrame) -> dict:
    """
    Derive scalar summary metrics from session-level aggregates.

    Metrics
    -------
    bounce_rate          : fraction of sessions with ≤ 1 page view.
    avg_session_duration : mean total seconds per session (timedelta-aware).
    avg_pages_per_session: mean total page views per session.
    total_sessions       : distinct session count.
    median_session_duration : median is more robust to extreme outliers.

    BUG (original) – three separate errors
    ----------------------------------------
    1.  bounce_rate used groupby().apply(lambda x: x == 1).mean() which
        returned a Series of booleans, not a scalar – mean() on a Series of
        booleans is undefined at the summary level and gives a per-row result
        rather than a fleet-level rate.
    2.  avg_session_time used groupby().sum() → returned a Series, not a
        scalar, so the metrics dict held a Series value.
    3.  pages_per_session used groupby().mean() on raw rows – meaning each row's
        page_views was averaged, not the per-session total.

    FIX: all three computed from a pre-aggregated session_metrics DataFrame
         where every statistic is already at session granularity.
    """

    # --- Bounce rate ---
    # A bounce = session with ≤ 1 total page view (0 view sessions are also
    # bounces, which the original eq(1) check missed entirely).
    bounce_rate = session_metrics["total_page_views"].le(1).mean()

    # --- avg session duration via timedelta for semantic clarity ---
    # Using timedelta makes the unit explicit and allows downstream formatting.
    mean_seconds = session_metrics["total_duration"].mean()
    avg_session_td = timedelta(seconds=float(mean_seconds))

    median_seconds = session_metrics["total_duration"].median()
    median_session_td = timedelta(seconds=float(median_seconds))

    return {
        "bounce_rate":              round(float(bounce_rate), ROUND_DIGITS),
        "avg_session_duration_sec": round(float(mean_seconds), 2),
        "avg_session_duration":     str(avg_session_td),          # HH:MM:SS
        "median_session_duration":  str(median_session_td),
        "avg_pages_per_session":    round(
            float(session_metrics["total_page_views"].mean()), ROUND_DIGITS
        ),
        "total_sessions":           int(session_metrics.shape[0]),
    }


def _compute_device_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """
    Produce per-device-type engagement metrics.

    BUG (original): used 'session_id': 'count' which counts rows, not
    distinct sessions.  A session spanning multiple rows would inflate counts.

    FIX: 'session_id': 'nunique' counts each session once.

    Enhancement: add avg_pages_per_session per device for a fair comparison
    with the global summary metric.
    """
    device_metrics = df.groupby("device_type", sort=True).agg(
        unique_sessions=("session_id", "nunique"),
        avg_duration_sec=("duration", "mean"),
        total_page_views=("page_views", "sum"),
    )

    # Per-device avg pages per session  (avoids division by zero)
    device_metrics["avg_pages_per_session"] = np.where(
        device_metrics["unique_sessions"] > 0,
        device_metrics["total_page_views"] / device_metrics["unique_sessions"],
        0.0,
    ).round(ROUND_DIGITS)

    device_metrics["avg_duration_sec"] = device_metrics["avg_duration_sec"].round(2)

    return device_metrics


# ---------------------------------------------------------------------------
# SECTION 4 – DASHBOARD VISUALISATION
# ---------------------------------------------------------------------------
def _build_dashboard(
    summary: dict,
    device_df: pd.DataFrame,
    session_df: pd.DataFrame,
    save_path: str | None = None,
) -> plt.Figure:
    """
    Render a single-figure summary dashboard containing:

    ┌────────────────────────────────────────────────────────────┐
    │  KPI cards row  (4 headline numbers)                       │
    ├──────────────────────┬─────────────────────────────────────┤
    │  Grouped bar chart   │  Session duration histogram         │
    │  (device breakdown)  │                                     │
    ├──────────────────────┴─────────────────────────────────────┤
    │  Bounce-rate gauge   │  Pages-per-session box-and-whisker  │
    └────────────────────────────────────────────────────────────┘

    A single composite figure was chosen over separate plots because it lets
    stakeholders read the full engagement story in one screenshot.
    """

    # ---- layout ----
    fig = plt.figure(figsize=(16, 10), facecolor="#0F172A")
    gs = gridspec.GridSpec(
        3, 4,
        figure=fig,
        hspace=0.55,
        wspace=0.35,
        top=0.92, bottom=0.07, left=0.06, right=0.97,
    )

    ACCENT   = "#38BDF8"   # sky-400
    ACCENT2  = "#818CF8"   # indigo-400
    ACCENT3  = "#34D399"   # emerald-400
    WARN     = "#FB923C"   # orange-400
    BG_CARD  = "#1E293B"
    FG       = "#F1F5F9"
    FG_DIM   = "#94A3B8"
    PALETTE  = [ACCENT, ACCENT2, ACCENT3, WARN, "#F472B6"]

    plt.rcParams.update({
        "text.color": FG,
        "axes.labelcolor": FG_DIM,
        "xtick.color": FG_DIM,
        "ytick.color": FG_DIM,
        "axes.edgecolor": "#334155",
        "font.family": "monospace",
    })

    def card_ax(gs_slice):
        ax = fig.add_subplot(gs_slice)
        ax.set_facecolor(BG_CARD)
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_edgecolor("#334155")
        return ax

    # ── ROW 0 : KPI cards ───────────────────────────────────────
    kpis = [
        ("Total Sessions",          f"{summary['total_sessions']:,}",        ACCENT),
        ("Bounce Rate",             f"{summary['bounce_rate']*100:.1f} %",   WARN),
        ("Avg Pages / Session",     f"{summary['avg_pages_per_session']:.2f}", ACCENT3),
        ("Avg Session Duration",    summary["avg_session_duration"],          ACCENT2),
    ]
    for i, (label, value, color) in enumerate(kpis):
        ax = card_ax(gs[0, i])
        ax.text(0.5, 0.62, value, ha="center", va="center",
                fontsize=22, fontweight="bold", color=color, transform=ax.transAxes)
        ax.text(0.5, 0.25, label, ha="center", va="center",
                fontsize=9, color=FG_DIM, transform=ax.transAxes)

    # ── ROW 1 LEFT : Device grouped bar chart ───────────────────
    ax_bar = fig.add_subplot(gs[1, :2])
    ax_bar.set_facecolor(BG_CARD)
    devices   = device_df.index.tolist()
    x         = np.arange(len(devices))
    bar_w     = 0.28

    bars_sess = ax_bar.bar(x - bar_w, device_df["unique_sessions"],
                           width=bar_w, color=ACCENT,  label="Unique Sessions", zorder=3)
    bars_pgs  = ax_bar.bar(x,          device_df["avg_pages_per_session"],
                           width=bar_w, color=ACCENT3, label="Avg Pages/Session", zorder=3)
    bars_dur  = ax_bar.bar(x + bar_w, device_df["avg_duration_sec"] / 60,
                           width=bar_w, color=ACCENT2, label="Avg Duration (min)", zorder=3)

    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels(devices, fontsize=9)
    ax_bar.set_title("Device Breakdown", color=FG, fontsize=11, pad=8)
    ax_bar.legend(fontsize=7, facecolor=BG_CARD, edgecolor="#334155", labelcolor=FG_DIM)
    ax_bar.grid(axis="y", color="#334155", linewidth=0.5, zorder=0)
    ax_bar.tick_params(colors=FG_DIM)

    # ── ROW 1 RIGHT : Session-duration histogram ─────────────────
    ax_hist = fig.add_subplot(gs[1, 2:])
    ax_hist.set_facecolor(BG_CARD)

    # Build session-level durations for histogram
    dur_vals = session_df["total_duration"].clip(upper=session_df["total_duration"].quantile(0.99))
    ax_hist.hist(dur_vals, bins=30, color=ACCENT2, alpha=0.85, edgecolor="#0F172A", zorder=3)
    ax_hist.axvline(session_df["total_duration"].mean(),
                    color=WARN, linewidth=1.5, linestyle="--", label="Mean", zorder=4)
    ax_hist.axvline(session_df["total_duration"].median(),
                    color=ACCENT3, linewidth=1.5, linestyle=":", label="Median", zorder=4)
    ax_hist.set_title("Session Duration Distribution (s)", color=FG, fontsize=11, pad=8)
    ax_hist.set_xlabel("Duration (seconds)", color=FG_DIM, fontsize=9)
    ax_hist.legend(fontsize=8, facecolor=BG_CARD, edgecolor="#334155", labelcolor=FG_DIM)
    ax_hist.grid(axis="y", color="#334155", linewidth=0.5, zorder=0)

    # ── ROW 2 LEFT : Bounce-rate donut / gauge ───────────────────
    ax_donut = fig.add_subplot(gs[2, :2])
    ax_donut.set_facecolor(BG_CARD)
    ax_donut.set_aspect("equal")

    br  = summary["bounce_rate"]
    wedge_colors = [WARN, "#334155"]
    ax_donut.pie(
        [br, 1 - br],
        colors=wedge_colors,
        startangle=90,
        wedgeprops=dict(width=0.4, edgecolor="#0F172A", linewidth=2),
        counterclock=False,
    )
    ax_donut.text(0, 0, f"{br*100:.1f}%", ha="center", va="center",
                  fontsize=20, fontweight="bold", color=WARN)
    ax_donut.set_title("Bounce Rate", color=FG, fontsize=11, pad=8)

    # ── ROW 2 RIGHT : Pages-per-session box plot ─────────────────
    ax_box = fig.add_subplot(gs[2, 2:])
    ax_box.set_facecolor(BG_CARD)

    if len(devices) > 0:
        box_data   = [
            df_group["total_page_views"].values
            for _, df_group in session_df.groupby("device_type_session")
            if len(df_group) > 0
        ] if "device_type_session" in session_df.columns else [session_df["total_page_views"].values]
        box_labels = session_df.get("device_type_session", pd.Series(["All"])).unique().tolist() \
                     if "device_type_session" in session_df.columns else ["All Sessions"]
    else:
        box_data   = [session_df["total_page_views"].values]
        box_labels = ["All Sessions"]

    bp = ax_box.boxplot(
        box_data,
        labels=box_labels,
        patch_artist=True,
        medianprops=dict(color=WARN, linewidth=2),
        whiskerprops=dict(color=FG_DIM),
        capprops=dict(color=FG_DIM),
        flierprops=dict(marker="o", color=ACCENT, markersize=3, alpha=0.5),
    )
    for patch, color in zip(bp["boxes"], PALETTE):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
        patch.set_edgecolor("#0F172A")

    ax_box.set_title("Pages per Session Distribution", color=FG, fontsize=11, pad=8)
    ax_box.set_xlabel("Device Type", color=FG_DIM, fontsize=9)
    ax_box.grid(axis="y", color="#334155", linewidth=0.5, zorder=0)

    # ── Main title ───────────────────────────────────────────────
    fig.suptitle(
        "User Engagement Summary Dashboard",
        color=FG, fontsize=15, fontweight="bold", y=0.97,
    )

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())

    return fig


# ---------------------------------------------------------------------------
# SECTION 5 – PUBLIC API
# ---------------------------------------------------------------------------
def analyze_user_engagement(
    logs_df: pd.DataFrame,
    show_dashboard: bool = True,
    save_path: str | None = None,
) -> tuple[dict, pd.DataFrame, plt.Figure | None]:
    """
    Analyse user engagement from a raw event-log DataFrame.

    Parameters
    ----------
    logs_df        : Raw event log.  Required columns:
                     session_id, page_views, duration, device_type.
                     Optional: start_time, end_time (ISO-8601 strings or
                     datetime objects) for timedelta-based duration derivation.
    show_dashboard : If True, render the matplotlib figure inline / to screen.
    save_path      : Optional file path (PNG/PDF) to save the dashboard.

    Returns
    -------
    summary_metrics : dict  – scalar KPIs for the entire dataset.
    device_metrics  : DataFrame – per-device breakdown.
    fig             : matplotlib Figure (or None if show_dashboard=False
                      and save_path is None).

    Raises
    ------
    TypeError  : if logs_df is not a DataFrame.
    ValueError : if required columns are missing or the DataFrame is empty.
    """
    # ── 1. Validate ─────────────────────────────────────────────
    df = _validate_inputs(logs_df)

    # ── 2. Clean ────────────────────────────────────────────────
    df = _clean_data(df)

    # ── 3. Session-level aggregation ────────────────────────────
    # Aggregate to one row per session before computing any metric.
    # This is the root fix for the original avg_session_time and
    # pages_per_session bugs (both were operating on row-level data).
    session_metrics = (
        df.groupby("session_id", sort=False)
        .agg(
            total_page_views=("page_views", "sum"),
            total_duration  =("duration",   "sum"),
            # Carry device type through for per-device box plot
            device_type_session=("device_type", "first"),
        )
        .reset_index()
    )

    # ── 4. Compute metrics ───────────────────────────────────────
    summary_metrics = _compute_summary_metrics(session_metrics)
    device_metrics  = _compute_device_metrics(df)

    # ── 5. Dashboard ─────────────────────────────────────────────
    fig = None
    if show_dashboard or save_path:
        fig = _build_dashboard(summary_metrics, device_metrics, session_metrics, save_path)
        if show_dashboard:
            plt.show()

    return summary_metrics, device_metrics, fig


# ---------------------------------------------------------------------------
# SECTION 6 – DEMO / SMOKE TEST
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    rng = np.random.default_rng(42)
    n   = 500

    # Build realistic synthetic data including some edge cases
    base_time = datetime(2024, 1, 1, 9, 0, 0)
    durations_sec = rng.integers(10, 1800, size=n).astype(float)
    start_times   = [base_time + timedelta(seconds=int(rng.integers(0, 86400))) for _ in range(n)]
    end_times     = [s + timedelta(seconds=int(d)) for s, d in zip(start_times, durations_sec)]

    synthetic = pd.DataFrame({
        "session_id":  rng.integers(1, 120, size=n),
        "page_views":  rng.integers(0, 15, size=n).astype(float),
        "duration":    durations_sec,
        "device_type": rng.choice(["mobile", "desktop", "tablet"], size=n),
        "start_time":  start_times,
        "end_time":    end_times,
    })

    # Inject intentional edge cases to test robustness
    synthetic.loc[0:4,   "page_views"] = np.nan     # missing page views
    synthetic.loc[5:9,   "duration"]   = -99         # negative durations
    synthetic.loc[10:14, "page_views"] = "bad_data"  # non-numeric strings

    summary, devices, figure = analyze_user_engagement(
        synthetic, show_dashboard=True, save_path="engagement_dashboard.png"
    )

    print("\n── Summary Metrics ──────────────────────────────────")
    for k, v in summary.items():
        print(f"  {k:<30} {v}")

    print("\n── Device Metrics ───────────────────────────────────")
    print(devices.to_string())

- SUMMARISED TECHNICAL EXPLANATION FROM THE AI

1. Bug 1 — Bounce Rate: Wrong logic and wrong result type
The original calculation was comparing page views row by row instead of at the session level, producing a raw list of true/false values rather than a single percentage. It also only flagged sessions with exactly one page view as bounces, completely ignoring sessions where a user landed and left without viewing any page at all. The fix aggregates everything to the session level first, then counts any session with one or fewer page views as a bounce, returning a single clean percentage.

2. Bug 2 — Average Session Time: Returned a list, not a number
Instead of computing a single average duration, the original code returned the total duration for every individual session as a list. This silently stored a whole table of numbers where a single value was expected, breaking anything that tried to display or use it downstream. The fix computes the true mean across all sessions and expresses it as a proper hours-minutes-seconds value so the unit is always unambiguous.

3. Bug 3 — Pages Per Session: Averaging at the wrong level
The original code calculated the average page views per row rather than per session, which gives a completely different — and wrong — number when sessions span multiple rows. The fix first totals the page views for each session, then averages those session totals, which is the correct calculation.

4. Bug 4 — Device Metrics: Counting rows instead of sessions
The device breakdown was counting the number of rows belonging to each device type rather than the number of distinct sessions. A single long session recorded across many rows would be counted as many sessions. The fix counts each unique session ID once, regardless of how many rows represent it.
Validation and Robustness

- Beyond the calculation bugs, the original had no defences against bad data at all. The fixes add checks for the wrong input type, an empty dataset, and missing columns — all raising clear descriptive errors. Non-numeric values in numeric columns are quietly converted to nulls rather than crashing the analysis. Negative durations and page-view counts trigger a warning with an exact count of affected rows, and those values are treated as missing rather than killing the entire run.

DISCUSSION QUESTIONS

1. HOW DID DIFFERENT PROMPTS HANDLE ERROR IDENTIFICATION

- In prompt 1, the code treats all errors as fatal and raises immediately, which is safe but brittle. 
- In prompt 2, the code distinguishes between structural errors (wrong type, missing columns) and data-quality errors (negatives, bad dates, non-numeric strings), handling each class appropriately and telling the caller exactly what was found and what was done about it.

#the further detail is included in the above segment which further explains how AI fixed the individual bugs

2. WHAT DATA VALIDATION METHODS WERE SUGGESTED?

- Type checking
Confirmed the input is actually a pandas DataFrame before touching it.

- Empty Data Check
Confirmed the DataFrame has at least one row before attempting any calculations.

- Numeric Correction
Forced page_views and duration columns to be numeric, turning any text or mixed values silently into nulls instead of crashing.

- Negative Value Guard
Scanned both numeric columns for negative numbers, issued a warning showing exactly how many were found, and replaced them with nulls so they don't skew results.

- Null implementation
Filled any remaining null values in page_views and duration with zero as a safe neutral default after all other checks passed.




3. HOW WAS TIME HANDLING IMPROVED

- No Time Handling Originally

The original code treated duration as just a plain number with no unit, no validation, and no awareness that it represented time at all.

- Datetime Parsing Added

Start and end time columns are now converted to proper datetime objects so Python understands them as actual points in time rather than plain text strings.

- Graceful Handling of Bad Dates

Any date string that cannot be parsed is converted to a null value instead of crashing the entire function, keeping the rest of the data usable.

- Bad Date Count Surfaced

After parsing, the code counts how many dates failed and issues a warning with that exact number so the caller knows the scale of the date quality problem.

- Timedelta Used for Duration Calculation

Where a raw duration number is missing, the code now calculates it by subtracting the start time from the end time using timedelta, which is mathematically correct and unit-safe.

- Negative Derived Durations Blocked

If subtracting timestamps somehow produces a negative duration, that value is rejected and treated as missing rather than being used in calculations.

- Duration Now Has an Explicit Unit

By wrapping the average session time in a timedelta object, the result is always expressed in hours, minutes, and seconds rather than a raw number whose unit could be seconds, milliseconds, or anything else.

- Mean and Median Both Expressed as Timedelta

Both the average and median session durations are converted to timedelta so every time-based output in the summary speaks the same unit-safe language consistently.

- SCENARIO 3
PROMPT

You are a data analyst writing SQL for a product analytics team.

We need to segment customers for a new feature rollout using the following schema:

Tables:

user_activity
- user_id
- last_login_date
- feature_usage_count
- account_type

transactions
- transaction_id
- user_id
- transaction_date
- amount
- platform

user_preferences
- user_id
- communication_preference
- interface_theme
- notification_settings

Task:

Write a single SQL query (using CTEs if needed) to:

1. Identify active users who have logged in within the last 30 days.
2. Calculate total spending per user from the transactions table.
3. Filter and return only high-value customers defined as the top 20% of users by total spending.
4. From this filtered group, join user_preferences and return insights into preference trends.

Output should include:
- user_id
- total_spent
- communication_preference
- interface_theme
- notification_settings

Constraints:
- Use clear and efficient SQL (CTEs preferred).
- Ensure correct aggregation of spending per user.
- Handle ranking correctly for top 20% segmentation.
- Assume standard SQL syntax (PostgreSQL-style unless otherwise specified).

Also briefly explain the logic of your query after writing it.




In [ ]:
WITH active_users AS (
    SELECT
        user_id
    FROM user_activity
    WHERE last_login_date >= CURRENT_DATE - INTERVAL '30 days'
),

user_spending AS (
    SELECT
        user_id,
        SUM(amount) AS total_spent
    FROM transactions
    GROUP BY user_id
),

active_user_spending AS (
    SELECT
        a.user_id,
        COALESCE(s.total_spent, 0) AS total_spent
    FROM active_users a
    LEFT JOIN user_spending s
        ON a.user_id = s.user_id
),

ranked_users AS (
    SELECT
        user_id,
        total_spent,
        NTILE(5) OVER (ORDER BY total_spent DESC) AS spending_quintile
    FROM active_user_spending
),

top_20_percent AS (
    SELECT
        user_id,
        total_spent
    FROM ranked_users
    WHERE spending_quintile = 1
)

SELECT
    t.user_id,
    t.total_spent,
    p.communication_preference,
    p.interface_theme,
    p.notification_settings
FROM top_20_percent t
JOIN user_preferences p
    ON t.user_id = p.user_id;


- # WHAT WORKS WELL

# Structure and Readability

The CTEs break the problem into clean, named steps that are easy to follow and debug individually.
Each CTE does exactly one job, which is good practice.

# Active User Filter

Using CURRENT_DATE - INTERVAL '30 days' is the correct PostgreSQL approach and will always calculate the window dynamically.

# Spending Aggregation

Grouping transactions by user_id and summing amount is correctly done before any joining, avoiding any risk of duplicate row inflation.

# COALESCE Usage

Wrapping total_spent in COALESCE(s.total_spent, 0) is a good defensive move, ensuring active users with zero transactions still appear with a 0 rather than a null.

# Top 20% Segmentation

NTILE(5) is the right function for this job and filtering on quintile 1 correctly isolates the top 20% by spending.

# Final Join

Joining only at the last step on the already-filtered group is efficient and clean.

# POTENTIAL CONCERNS
- NTILE behaviour on small datasets

- If the active user pool is small, say fewer than 5 users, NTILE will still attempt to divide them into 5 buckets and the results can be uneven or misleading.
For large production datasets this is not a concern, but worth noting if the query runs on filtered subsets during testing.



# FOLLOWUP PROMPTS

- You are a PostgreSQL database engineer. Take the SQL query below and make exactly two changes: 
add early transaction filtering inside the user_spending CTE and add the recommended indexes as commented lines at the top. Do not change anything else in the query. Use percentile instead of NTILE. After the code, write a short plain English explanation of what each change does and why it improves performance. Keep the explanation simple, no code examples in the explanation.

In [ ]:
-- Recommended indexes to add before running the query
-- CREATE INDEX idx_user_activity_login ON user_activity(user_id, last_login_date);
-- CREATE INDEX idx_transactions_user_date ON transactions(user_id, transaction_date);
-- CREATE INDEX idx_user_preferences_user ON user_preferences(user_id);

WITH active_users AS (
    SELECT
        user_id
    FROM user_activity
    WHERE last_login_date >= CURRENT_DATE - INTERVAL '30 days'
),
user_spending AS (
    SELECT
        t.user_id,
        SUM(t.amount) AS total_spent
    FROM transactions t
    WHERE t.transaction_date >= CURRENT_DATE - INTERVAL '90 days'
    AND t.user_id IN (SELECT user_id FROM active_users)
    GROUP BY t.user_id
),
active_user_spending AS (
    SELECT
        a.user_id,
        COALESCE(s.total_spent, 0) AS total_spent
    FROM active_users a
    LEFT JOIN user_spending s
        ON a.user_id = s.user_id
),
ranked_users AS (
    SELECT
        user_id,
        total_spent,
        -- PERCENT_RANK gives a precise 0 to 1 score for each user
        -- filtering above 0.8 guarantees a true mathematical top 20%
        PERCENT_RANK() OVER (ORDER BY total_spent DESC) AS spend_percentile
    FROM active_user_spending
),
top_20_percent AS (
    SELECT
        user_id,
        total_spent
    FROM ranked_users
    -- PERCENT_RANK scores from 0 (highest) to 1 (lowest)
    -- so values below 0.2 represent the top 20%
    WHERE spend_percentile < 0.2
)
SELECT
    t.user_id,
    t.total_spent,
    p.communication_preference,
    p.interface_theme,
    p.notification_settings
FROM top_20_percent t
JOIN user_preferences p
    ON t.user_id = p.user_id;

# EXPLANATION FROM THE AI 


# What Changed and Why

# Early Transaction Filtering

- The user_spending CTE now filters transactions in two ways before doing any aggregation.
- First it limits transactions to the last 90 days, so historical spending from years ago no longer inflates a user's total.
- Second it restricts the aggregation to only users who are already confirmed as active, meaning the database never touches transactions belonging to inactive users at all.
Together these two filters dramatically reduce the number of rows the database has to sum before any joining or ranking happens.

# Indexes

- Three index suggestions have been added as commented lines at the top so they are easy to find and run once on the database.
- The first index covers user_id and last_login_date together because the active user filter uses both columns.
- The second index covers user_id and transaction_date on the transactions table because both columns are now used in the early filter.
- The third index covers user_id on the preferences table to speed up the final join.
- These indexes mean the database can jump directly to the relevant rows instead of scanning entire tables on every run. 

# PERCENT_RANK replacing NTILE

- NTILE works by dividing rows into equal sized buckets, so if you have 103 users it forces them into 5 groups of unequal size and the boundaries between groups can land in arbitrary places.
- PERCENT_RANK instead gives every user a precise score between 0 and 1 based on their actual position in the spending distribution, where 0 is the highest spender and 1 is the lowest.
- Filtering where that score is below 0.2 means you are always getting a mathematically exact top 20% regardless of how many users are in the dataset.
- This matters most on smaller or unevenly distributed datasets where NTILE's bucket boundaries could accidentally include or exclude borderline users.

# DISCUSSION

# 1. HOW WERE PERCENTILE CALCULATIONS HANDLED

- In the very first version of the code, there was no percentile handling at all. It simply counted rows per device or session with no ranking logic whatsoever. When the SQL query was introduced, NTILE was used to split users into 5 equal buckets and anyone landing in bucket 1 was labelled top 20%, which works but can place borderline users in the wrong group when the dataset does not divide evenly. The final version replaced this with PERCENT_RANK, which gives every user a precise score between 0 and 1 and filters on that score directly, making the top 20% boundary mathematically exact every time.

# 2. WHAT APPROACHES TO DATE FILTERING WERE USED?

- Two approaches to date filtering were used across the conversation.
The first was a simple activity window check, where CURRENT_DATE - INTERVAL '30 days' was used to confirm a user had logged in recently. This is a dynamic calculation that updates automatically every time the query runs so no hardcoded dates are needed.
The second was a transaction date filter added during the optimisation stage, where CURRENT_DATE - INTERVAL '90 days' was applied directly inside the transaction aggregation. This was wider than the login window deliberately, giving enough historical spending data to make the high-value segmentation meaningful while still excluding very old transactions that would distort the results.
The key improvement across the conversation was that the first version had no transaction date filter at all, meaning spending was calculated from the entire history of the table. Adding the second filter aligned the spending calculation with a relevant time window, making the two date boundaries consistent with each other and the segmentation far more accurate.

# 3. HOW WAS THE QUERY OPTIMISED?

The query was optimised in three specific ways across the conversation.

# Early Transaction Filtering

- The original query aggregated every single transaction in the entire table first and then filtered down to active users afterwards. This meant the database was doing the heaviest work on the full dataset before any narrowing had happened. The fix moved the filtering inside the transaction aggregation step so the database only ever processes transactions belonging to active users within the last 90 days, dramatically reducing the number of rows touched from the very start.


# Indexes Added

- The original query had no indexing guidance at all, meaning the database would perform a full table scan on every run. Three indexes were added as commented lines at the top covering user_id and last_login_date on the activity table, user_id and transaction_date on the transactions table, and user_id on the preferences table. These allow the database to jump directly to the relevant rows instead of reading entire tables each time the query runs.


# Ranking Method Improved
- NTILE was replaced with PERCENT_RANK which is a more precise and computationally cleaner way to identify the top 20%. NTILE forces rows into equal buckets which can cause uneven splits and misclassify borderline users, adding unnecessary complexity to how the database handles the ranking. PERCENT_RANK assigns each user a clean decimal score and the filter becomes a simple mathematical boundary, making the operation both more accurate and more straightforward to execute.